In [24]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/smart-mcq-solver-challenge/sample_submission.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv
/kaggle/input/competitions/smart-mcq-solver-challenge/test.csv


## Q1.

In [25]:
label_mapping = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

In [26]:
import pandas as pd

train = pd.read_csv("/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")

label_mapping = {
    "A": 0,
    "B": 1,
    "C": 2,
    "D": 3,
    "E": 4
}

train["label"] = train["answer"].map(label_mapping)

In [27]:
encoded_label = train.loc[150, "label"]
print(encoded_label)

2


## Q2.

In [28]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2000 entries, 0 to 1999
Data columns (total 9 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   id      2000 non-null   int64 
 1   prompt  2000 non-null   object
 2   A       2000 non-null   object
 3   B       2000 non-null   object
 4   C       2000 non-null   object
 5   D       2000 non-null   object
 6   E       2000 non-null   object
 7   answer  2000 non-null   object
 8   label   2000 non-null   int64 
dtypes: int64(2), object(7)
memory usage: 140.8+ KB


In [29]:
formatted_input = str(train.loc[0, "prompt"]) + " [SEP] " + str(train.loc[0, "B"])

print(formatted_input)
print("Length:", len(formatted_input))

Pick the best possible answer: What is Martin Heidegger's view on the relationship between time and human existence? among the listed options. [SEP] Martin Heidegger believes that humans do not exist inside time, but that they are time. The relationship to the past is a present awareness of having been, and the relationship to the future involves anticipating a potential possibility, task, or engagement.
Length: 407


## Q3.

In [30]:
from transformers import AutoTokenizer
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

row = train.loc[0]

choices = [
    str(row["prompt"]) + " [SEP] " + str(row["A"]),
    str(row["prompt"]) + " [SEP] " + str(row["B"]),
    str(row["prompt"]) + " [SEP] " + str(row["C"]),
    str(row["prompt"]) + " [SEP] " + str(row["D"]),
    str(row["prompt"]) + " [SEP] " + str(row["E"]),
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt",
)

input_ids = encoding["input_ids"].unsqueeze(0)

print(input_ids.shape)

torch.Size([1, 5, 128])


## Q4.

In [31]:
total_token_positions = 16 * 5 * 128
print(total_token_positions)

10240


## Q5.

In [32]:
from transformers import AutoTokenizer, AutoModelForMultipleChoice
import torch

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

row = train.loc[0]

choices = [
    f"{row['prompt']} [SEP] {row['A']}",
    f"{row['prompt']} [SEP] {row['B']}",
    f"{row['prompt']} [SEP] {row['C']}",
    f"{row['prompt']} [SEP] {row['D']}",
    f"{row['prompt']} [SEP] {row['E']}",
]

encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

inputs = {k: v.unsqueeze(0) for k, v in encoding.items()}
outputs = model(**inputs)

print(outputs.logits.shape)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


torch.Size([1, 5])


## Q6.

In [33]:
import torch
from transformers import AutoTokenizer, AutoModelForMultipleChoice

# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModelForMultipleChoice.from_pretrained("bert-base-uncased")

# Get row 0
row = train.loc[0]

# Create 5 prompt-option pairs
choices = [
    f"{row['prompt']} [SEP] {row['A']}",
    f"{row['prompt']} [SEP] {row['B']}",
    f"{row['prompt']} [SEP] {row['C']}",
    f"{row['prompt']} [SEP] {row['D']}",
    f"{row['prompt']} [SEP] {row['E']}",
]

# Tokenize
encoding = tokenizer(
    choices,
    padding="max_length",
    truncation=True,
    max_length=128,
    return_tensors="pt"
)

# Reshape to [1, 5, 128]
input_ids = encoding["input_ids"].unsqueeze(0)
attention_mask = encoding["attention_mask"].unsqueeze(0)

# If token_type_ids exist (BERT has them)
inputs = {
    "input_ids": input_ids,
    "attention_mask": attention_mask
}

if "token_type_ids" in encoding:
    inputs["token_type_ids"] = encoding["token_type_ids"].unsqueeze(0)

# Correct label
labels = torch.tensor([train.loc[0, "label"]])

# Forward pass
outputs = model(**inputs, labels=labels)

print("Loss:", outputs.loss)
print("Loss shape:", outputs.loss.shape)
print("Loss dimensions:", outputs.loss.dim())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Loss: tensor(1.6178, grad_fn=<NllLossBackward0>)
Loss shape: torch.Size([])
Loss dimensions: 0


## Q7.

In [34]:
!pip uninstall -y transformers peft torchao accelerate tokenizers
!pip install -q \
    transformers==4.52.4 \
    peft==0.15.2 \
    accelerate==1.7.0 \
    datasets \
    sentencepiece

Found existing installation: transformers 4.52.4
Uninstalling transformers-4.52.4:
  Successfully uninstalled transformers-4.52.4
Found existing installation: peft 0.15.2
Uninstalling peft-0.15.2:
  Successfully uninstalled peft-0.15.2
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
Found existing installation: tokenizers 0.21.4
Uninstalling tokenizers-0.21.4:
  Successfully uninstalled tokenizers-0.21.4
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.1/362.1 kB 2.1 MB/s eta 0:00:00a 0:00:01


In [35]:
import transformers
import peft
import torch

print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("Torch:", torch.__version__)

try:
    import torchao
    print("TorchAO:", torchao.__version__)
except ImportError:
    print("TorchAO: Not Installed ✅")

Transformers: 5.0.0
PEFT: 0.19.1
Torch: 2.10.0+cpu
TorchAO: 0.10.0


In [36]:
from transformers import AutoModelForMultipleChoice

model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

print(type(model))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


<class 'transformers.models.bert.modeling_bert.BertForMultipleChoice'>


In [37]:
from peft import LoraConfig, get_peft_model, TaskType

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(model, lora_config)

In [38]:
trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(f"Trainable Parameters: {trainable_params:,}")
print(f"Total Parameters: {total_params:,}")
print(f"Percentage: {100 * trainable_params / total_params:.4f}%")

Trainable Parameters: 295,681
Total Parameters: 109,778,690
Percentage: 0.2693%


## Q8.

In [39]:
from datasets import Dataset
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

# First 100 rows
train_100 = train.iloc[:100].copy()

def preprocess(example):
    choices = [
        f"{example['prompt']} [SEP] {example['A']}",
        f"{example['prompt']} [SEP] {example['B']}",
        f"{example['prompt']} [SEP] {example['C']}",
        f"{example['prompt']} [SEP] {example['D']}",
        f"{example['prompt']} [SEP] {example['E']}",
    ]

    encoding = tokenizer(
        choices,
        padding="max_length",
        truncation=True,
        max_length=128,
    )

    return {
        "input_ids": encoding["input_ids"],          
        "attention_mask": encoding["attention_mask"],
        "labels": example["label"],
    }

dataset = Dataset.from_pandas(train_100)
dataset = dataset.map(preprocess)

print(len(dataset[0]["input_ids"]))
print(len(dataset[0]["input_ids"][0]))

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

5
128


In [43]:
print("model" in globals())
print("trainer" in globals())
print("train_dataset" in globals())

True
False
False


In [44]:
from transformers import AutoModelForMultipleChoice

model = AutoModelForMultipleChoice.from_pretrained(
    "bert-base-uncased"
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForMultipleChoice LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [46]:
from peft import LoraConfig, TaskType, get_peft_model

config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["query", "value"],
    lora_dropout=0.1,
    bias="none",
    task_type=TaskType.SEQ_CLS,
)

model = get_peft_model(model, config)

/usr/local/lib/python3.12/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/peft/tuners/tuners_utils.py:302: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [49]:
train_dataset = ...

In [50]:
training_args = TrainingArguments(
    output_dir="./outputs",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=1,
    max_steps=4,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
)

NameError: name 'TrainingArguments' is not defined

In [51]:
trainer.train()
print(trainer.state.global_step)

NameError: name 'trainer' is not defined

In [41]:
trainer.train()

print(trainer.state.global_step)

NameError: name 'trainer' is not defined

In [42]:
import torch
import torch.nn.functional as F

# Put model in evaluation mode
model.eval()

# Get the first example from your processed dataset
sample = tokenized_dataset[0]

inputs = {
    "input_ids": torch.tensor(sample["input_ids"]).unsqueeze(0),
    "attention_mask": torch.tensor(sample["attention_mask"]).unsqueeze(0),
}

# Include token_type_ids if your dataset contains them
if "token_type_ids" in sample:
    inputs["token_type_ids"] = torch.tensor(sample["token_type_ids"]).unsqueeze(0)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
probs = F.softmax(logits, dim=1)

print("Probabilities:", probs)
print("Probability of Option E:", probs[0, 4].item())
print("Rounded:", round(probs[0, 4].item(), 4))

NameError: name 'tokenized_dataset' is not defined

In [54]:
import torch
import torch.nn.functional as F

# Put model in evaluation mode
model.eval()

# Get first example
sample = train_dataset[0]

# Prepare inputs
inputs = {
    "input_ids": torch.tensor(sample["input_ids"]).unsqueeze(0),
    "attention_mask": torch.tensor(sample["attention_mask"]).unsqueeze(0),
}

# BERT also uses token_type_ids
if "token_type_ids" in sample:
    inputs["token_type_ids"] = torch.tensor(sample["token_type_ids"]).unsqueeze(0)

# Inference
with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits
probabilities = torch.softmax(logits, dim=1)

print("Logits:")
print(logits)

print("\nProbabilities:")
print(probabilities)

print("\nProbability of Option E:")
print(round(probabilities[0, 4].item(), 4))

TypeError: 'ellipsis' object is not subscriptable